# 02 — Baseline Training

Train the clean baseline (ResNet-50 and EfficientNet-B0) on the finalized 7,986-image manifest. All reusable logic lives in `src/`; this notebook only wires configuration, execution, and result inspection.

Methodology: clean train data → clean-trained model → validation-selected checkpoint. The test split is NOT touched here (see notebook 03). No robustness transforms in this notebook.

## 1. Configuration

Single place for every run knob. One model per run; switch `MODEL_NAME` for the second baseline.

In [ ]:
MODEL_NAME = "resnet50"  # or "efficientnet_b0"
EPOCHS = 10
BATCH_SIZE = 32
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
OPTIMIZER = "adam"
AUGMENT = False  # keep False for the interpretable clean baseline
PRETRAINED = True
SEED = 42
CHECKPOINT_DIR = "models/baseline"
print("config:", MODEL_NAME, f"epochs={EPOCHS}", f"batch={BATCH_SIZE}", f"augment={AUGMENT}")

## 2. Seed and device

CPU is the reproducibility reference; CUDA is used when available.

In [ ]:
import sys
sys.path.insert(0, "..")  # run from notebooks/; no-op otherwise
from src.train import get_device, set_seed
set_seed(SEED)
DEVICE = get_device()
print("device:", DEVICE)

## 3. Dataset loading

Manifest is the source of truth; train/val only — no test loader in the training path.

In [ ]:
from src.preprocessing import IMAGE_SIZE, get_transform
from src.train import make_dataloaders
train_loader, val_loader = make_dataloaders(
    batch_size=BATCH_SIZE, augment=AUGMENT, seed=SEED)
print("train batches:", len(train_loader), "val batches:", len(val_loader))
print("input size:", IMAGE_SIZE)

## 4. Dataset sanity checks

Expected: train 5,600 (2,800/2,800), val 1,200 (600/600). Fails loudly otherwise.

In [ ]:
from src.dataset import GenImageDataset
train_ds = GenImageDataset(split="train")
val_ds = GenImageDataset(split="val")
assert len(train_ds) == 5600 and len(val_ds) == 1200
assert train_ds.class_counts() == {0: 2800, 1: 2800}
assert val_ds.class_counts() == {0: 600, 1: 600}
xb, yb = next(iter(train_loader))
print("batch:", tuple(xb.shape), yb.dtype, "labels:", sorted(set(yb.tolist())))
print("sanity checks passed")

## 5. Model selection

One constructor for both baselines; 2-class head; no custom architecture.

In [ ]:
from src.model import build_model, count_parameters
model = build_model(MODEL_NAME, pretrained=PRETRAINED)
print(MODEL_NAME, "trainable params:", count_parameters(model))

## 6. Training

Runs the shared loop from `src/train.py`. This is the expensive step — expect minutes per epoch on GPU, much longer on CPU.

In [ ]:
from pathlib import Path
from src.train import TrainConfig, fit
config = TrainConfig(model_name=MODEL_NAME, pretrained=PRETRAINED,
    epochs=EPOCHS, batch_size=BATCH_SIZE, learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY, optimizer=OPTIMIZER, augment=AUGMENT,
    seed=SEED, checkpoint_dir=CHECKPOINT_DIR)
history = fit(config)  # writes best/last checkpoints + history JSON

## 7. Validation

Best epoch was selected on validation accuracy inside `fit()`; re-state it here for the record.

In [ ]:
print("best epoch:", history["best_epoch"], "best val acc:", round(history["best_val_acc"], 4))
print("final epoch val acc:", round(history["val_acc"][-1], 4))

## 8. Checkpoint path

The artifact that notebook 03 will load. Never retrain per transformation there.

In [ ]:
best_ckpt = Path(CHECKPOINT_DIR) / f"{MODEL_NAME}_best.pth"
print("best checkpoint:", best_ckpt, "exists:", best_ckpt.is_file())

## 9. Final test evaluation (placeholder — NOT RUN)

Runs in notebook 03 against the untouched test split (1,186 images), using `src/evaluate.py` on the checkpoint above. Do not evaluate here.

In [ ]:
# NOT RUN — see scripts/run_jpeg_robustness.py (and sibling run_* scripts)
# from src.evaluate import evaluate_model  # test evaluation lives there
print("test evaluation: NOT RUN")

## 10. Results logging (placeholder)

Copy real metrics here only after execution; never invent values. Routing: metrics rows → `results/metrics/`, figures → `results/graphs/`, narrative → `docs/RESULTS_REFERENCE.md`.

In [ ]:
# NOT RUN — fill after real training + test evaluation.
results = {"model": MODEL_NAME, "status": "NOT RUN"}
print(results)